# DMPJ 自己対戦RL（Colab 用）

このノートブックは DMPJ の自己対戦学習を Colab で大きく回すためのものです。
実行すると、リポジトリを取得し、自己対戦でモデルを学習し、学習前後の勝率
（対ランダム／対greedy）を表示します。結果（model.npz とログ）は最後のセルで
Google Drive に保存するか、ダウンロードできます。それを渡してもらえれば、
続きの検証と学習をこちらで行えます。

注意: 収集（対戦の実行）は CPU が律速なので、GPU ランタイムでなくても構いません。
torch が使える場合は将来の torch 版モデルの学習に使えます。

In [ ]:
# リポジトリ取得（AI 系ブランチ）
!git clone -q -b work/phase6-speed-attention https://github.com/KOBA1616/DMPJ.git
%cd DMPJ
!pip install -q pyyaml numpy scipy pytest

In [ ]:
# 環境確認（torch があればバージョンを表示）
import sys, numpy
print(sys.version)
print('numpy', numpy.__version__)
try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
except ImportError:
    print('torch なし（numpy モデルで学習します）')

In [ ]:
# 学習の実行（規模はここで調整。目安: 100 iter × 16局 = 1600局）
# 注意機構モデルにする場合は --model attn を付ける
!DMSIM_RUST=0 python tools/selfplay_train.py --iters 100 --games-per-iter 16 --eval-games 40 --opponents random,greedy --workers 2 --out /content/model_colab.npz 2>&1 | tee /content/selfplay_log.txt | tail -30

In [ ]:
# 結果の保存（Drive を使う場合）: マウントしてコピー
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/model_colab.npz /content/selfplay_log.txt /content/drive/MyDrive/

# Drive を使わない場合はダウンロード:
from google.colab import files
files.download('/content/model_colab.npz')
files.download('/content/selfplay_log.txt')